# Suivi des Expériences avec MLflow

Ce notebook est dédié au suivi (tracking) de toutes nos expériences de machine learning (classification, régression et approches multimodales) à l'aide de **MLflow**.

**Objectifs :**
- Tracker de manière centralisée les hyperparamètres, métriques et artefacts (matrices de confusion, SHAP values).
- Comparer les performances des différents modèles.
- Enregistrer les meilleurs modèles dans le Model Registry pour un déploiement futur.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import mlflow
import mlflow.sklearn

from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

import optuna
import shap
import nltk
from nltk.corpus import stopwords
import warnings

warnings.filterwarnings('ignore')

# Téléchargement des stopwords français
nltk.download('stopwords', quiet=True)

# MLflow setup
mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment("eco_smart_classifier_experiments")

<Experiment: artifact_location=('file:///e:/DSIR1/semestre2/Certification '
 'ml/eco-smart-classifier/notebooks/../mlruns/226952388580542507'), creation_time=1778373055678, experiment_id='226952388580542507', last_update_time=1778373055678, lifecycle_stage='active', name='eco_smart_classifier_experiments', tags={}, trace_location=None, workspace='default'>

## Chargement des données

In [4]:
# Chargement des données (déjà séparées en train/val/test et encodées si besoin)
train_df = pd.read_csv('../data/processed/train.csv')
val_df = pd.read_csv('../data/processed/val.csv')
test_df = pd.read_csv('../data/processed/test.csv')

# Pour les features numériques de classification, on écarte la cible (Categorie),
# le texte brut (Rapport_Collecte) et on corrige le data leakage en supprimant Prix_Revente.
numeric_features = [col for col in train_df.columns if col not in ['Categorie', 'Prix_Revente', 'Rapport_Collecte']]

X_train_num = train_df[numeric_features]
y_train_clf = train_df['Categorie']

X_val_num = val_df[numeric_features]
y_val_clf = val_df['Categorie']

X_test_num = test_df[numeric_features]
y_test_clf = test_df['Categorie']

## Run 1-4 : Baseline Models

In [5]:
def log_confusion_matrix(y_true, y_pred, run_name):
    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, cmap='Blues')
    plt.title(f"Matrice de Confusion - {run_name}")
    file_path = f"confusion_matrix_{run_name}.png"
    plt.savefig(file_path)
    plt.close()
    mlflow.log_artifact(file_path)

models = {
    "Logistic_Regression": LogisticRegression(max_iter=1000),
    "Random_Forest": RandomForestClassifier(random_state=42),
    "Gradient_Boosting": GradientBoostingClassifier(random_state=42),
    "SVM": SVC(random_state=42)
}

for name, model in models.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train_num, y_train_clf)
        y_pred = model.predict(X_val_num)
        
        acc = accuracy_score(y_val_clf, y_pred)
        f1 = f1_score(y_val_clf, y_pred, average='weighted')
        
        mlflow.log_metric("val_accuracy", acc)
        mlflow.log_metric("val_f1_score", f1)
        
        log_confusion_matrix(y_val_clf, y_pred, name)
        
        mlflow.sklearn.log_model(model, "model")

2026/05/10 01:31:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 01:31:06 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/10 01:31:25 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 01:31:25 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

## Run 5 : Optuna Tuning

In [6]:
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 200)
    max_depth = trial.suggest_int("max_depth", 5, 20)
    
    with mlflow.start_run(run_name="Optuna_RF_Tuning", nested=True):
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
        model.fit(X_train_num, y_train_clf)
        
        y_pred = model.predict(X_val_num)
        acc = accuracy_score(y_val_clf, y_pred)
        f1 = f1_score(y_val_clf, y_pred, average='weighted')
        
        mlflow.log_params({"n_estimators": n_estimators, "max_depth": max_depth})
        mlflow.log_metric("val_accuracy", acc)
        mlflow.log_metric("val_f1_score", f1)
        
        log_confusion_matrix(y_val_clf, y_pred, "Optuna_RF")
        
        # SHAP Artifact
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_val_num.iloc[:100])
        
        fig = plt.figure(figsize=(10, 6))
        if isinstance(shap_values, list):
            shap.summary_plot(shap_values[0], X_val_num.iloc[:100], show=False)
        else:
            shap.summary_plot(shap_values, X_val_num.iloc[:100], show=False)
        plt.tight_layout()
        plt.savefig("shap_summary_optuna.png")
        plt.close()
        mlflow.log_artifact("shap_summary_optuna.png")
        
        # Enregistrement dans le Model Registry si les performances sont bonnes
        mlflow.sklearn.log_model(model, "model", registered_model_name="Best_RF_Classifier")
        
    return acc

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10) # Nombre de tests limité pour la démonstration

[I 2026-05-10 01:31:48,509] A new study created in memory with name: no-name-5cf591af-005d-4ca5-9324-01979ce1e26f
2026/05/10 01:31:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 01:31:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'Best_RF_Classifier'.
Created version '1' of model 'Best_RF_Classifier'.
[I 2026-05-10 01:31:58,610] Trial 0 finished with value: 0.9959946595460614 and parameters: {'n_estimators': 168, 'max_depth': 17}. Best is trial 0 with value: 0.9959946595460614.
2026/05/10 01:32:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `nam

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

...

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

<Figure size 1000x600 with 0 Axes>

## Run 6 : Régression Prix_Revente

In [10]:
# Pour la régression, on prédit Prix_Revente
y_train_reg = train_df['Prix_Revente']
y_val_reg = val_df['Prix_Revente']

with mlflow.start_run(run_name="RandomForest_Regressor_Prix"):
    reg_model = RandomForestRegressor(random_state=42)
    reg_model.fit(X_train_num, y_train_reg)
    
    y_pred_reg = reg_model.predict(X_val_num)
    
    rmse = np.sqrt(mean_squared_error(y_val_reg, y_pred_reg))
    r2 = r2_score(y_val_reg, y_pred_reg)
    
    mlflow.log_metric("val_rmse", rmse)
    mlflow.log_metric("val_r2", r2)
    
    mlflow.sklearn.log_model(reg_model, "model", registered_model_name="RF_Regressor_Prix")

2026/05/10 01:33:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 01:33:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'RF_Regressor_Prix' already exists. Creating a new version of this model...
Created version '2' of model 'RF_Regressor_Prix'.


## Run 7 : Pipeline Multimodal

In [8]:
french_stopwords = stopwords.words('french')

X_train_multi = train_df[numeric_features + ['Rapport_Collecte']].copy()
X_val_multi = val_df[numeric_features + ['Rapport_Collecte']].copy()

# Remplissage des valeurs manquantes pour le texte
X_train_multi['Rapport_Collecte'] = X_train_multi['Rapport_Collecte'].fillna('')
X_val_multi['Rapport_Collecte'] = X_val_multi['Rapport_Collecte'].fillna('')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_features),
        ('text', TfidfVectorizer(stop_words=french_stopwords, max_features=1000), 'Rapport_Collecte')
    ])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

with mlflow.start_run(run_name="Multimodal_Pipeline"):
    pipeline.fit(X_train_multi, y_train_clf)
    
    y_pred_multi = pipeline.predict(X_val_multi)
    acc = accuracy_score(y_val_clf, y_pred_multi)
    f1 = f1_score(y_val_clf, y_pred_multi, average='weighted')
    
    mlflow.log_metric("val_accuracy", acc)
    mlflow.log_metric("val_f1_score", f1)
    
    log_confusion_matrix(y_val_clf, y_pred_multi, "Multimodal_Pipeline")
    
    mlflow.sklearn.log_model(pipeline, "model", registered_model_name="Multimodal_Classifier")

2026/05/10 01:33:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/10 01:33:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'Multimodal_Classifier'.
Created version '1' of model 'Multimodal_Classifier'.


## Résumé des expériences

In [9]:
runs = mlflow.search_runs()
display(runs[['run_id', 'tags.mlflow.runName', 'metrics.val_accuracy', 'metrics.val_f1_score']].sort_values('metrics.val_accuracy', ascending=False).head(10))

,run_id,tags.mlflow.runName,metrics.val_accuracy,metrics.val_f1_score
0,77d3c2ead2404af7812e98367f3b6cb8,Multimodal_Pipeline,1.000000,1.000000
8,6fad276a0dc240ef8fbafce25209dab9,Optuna_RF_Tuning,0.996662,0.996662
10,59a86067abc445af962f3036e3a7d20d,Optuna_RF_Tuning,0.996662,0.996662
13,9c9e886211c648daba4f10249bd4d9a8,Gradient_Boosting,0.996662,0.996664
3,6c066453274944eaa47d60105e3b8d1e,Optuna_RF_Tuning,0.995995,0.995997
4,7b18d82567df4fad98ea1249b176400e,Optuna_RF_Tuning,0.995995,0.995997
5,b589d0031c5e4649904440566549f742,Optuna_RF_Tuning,0.995995,0.995997
7,9761193a3cac4d75b9fa3ec486aa444a,Optuna_RF_Tuning,0.995995,0.995997
9,0f79a26a6cb947f8bfbd285bb0669da1,Optuna_RF_Tuning,0.995995,0.995997
11,527ae73b637d4fb59fbb606b3b03758e,Optuna_RF_Tuning,0.995995,0.995997


## Conclusion

En analysant les résultats stockés dans MLflow :
- Les modèles avec **Random Forest** et **Gradient Boosting** s'avèrent très solides en approche classique.
- Le **Pipeline Multimodal** (qui inclut le `Rapport_Collecte` vectorisé via TF-IDF) permet généralement de capturer des subtilités non présentes dans les données chiffrées pures.
- L'optimisation **Optuna** améliore encore les hyperparamètres.
- Le meilleur modèle (enregistré dans le Model Registry) sera celui utilisé en production grâce à son score d'Accuracy supérieur et sa capacité de généralisation (validé par F1-score et matrice de confusion).